In [ ]:
!npm install -g @openai/codex

In [ ]:
!codex --version

In [ ]:
!codex login --device-auth

In [ ]:
!mkdir -p /content/deepfake_project
%cd /content/deepfake_project

In [ ]:
%%writefile /content/deepfake_project/cnn_comparison_prompt.md
# 기존 GenD + RAG 프로젝트에 경량 CNN 비교 실험 추가

현재 Google Colab에서 다음 프로젝트를 구현한 상태입니다.

프로젝트 경로:
`/content/deepfake_project`

기존 노트북:
`/content/deepfake_project/GenD_RAG_Colab.ipynb`

Google Colab GPU:
- NVIDIA Tesla T4 (약 15GB VRAM)
- PyTorch 2.14.1+cu130
- CUDA 사용 가능: True
- GPU 연산 정상 확인

**목표는 기존 프로젝트를 유지하면서 일반 CNN 모델을 추가하고, 총 4개 모델의 이미지 분류 성능을 실제로 비교하는 것입니다.**

코드만 작성하는 것이 아니라 무료 Colab 환경에서 실행 가능한 가벼운 실험을 완성해 주세요.

## 1. 비교 모델

다음 4가지 모델을 구현하고 비교합니다.

**A. 일반 CNN**
- PyTorch로 구현한 가벼운 CNN
- Conv2d, BatchNorm, ReLU, MaxPooling, Linear 등을 사용
- 3~4개 Convolution Block
- 학습 이미지 크기 128×128
- 기본 학습 5 Epoch
- Batch Size 16 또는 32
- Adam Optimizer
- BCEWithLogitsLoss 또는 CrossEntropyLoss
- Validation 성능을 기준으로 최적 모델 저장
- GPU 메모리 부족 시 자동으로 Batch Size 축소

**B. GenD**
- 기존에 구현한 GenD 사전학습 모델 재사용
- 모델을 새로 학습하지 않음
- 공식 구현의 전처리와 출력 정의 유지
- Real/Fake 분류 점수 출력

**C. RAG 기반 분류**
- CLIP 이미지 임베딩 사용
- FAISS 유사도 검색
- Top-K 검색 결과를 이용해 Real/Fake 분류
- K=3 또는 K=5
- 유사도 기반 가중 투표 구현

**D. GenD + RAG**
- 기존 하이브리드 구현 재사용
- 검증 데이터에서 결정한 결합 가중치 사용
- 새로운 이미지에 대해 두 방법의 예측을 결합
- Exact Match와 Similarity Search 구분

## 2. 데이터셋

우선 다음 공개 데이터셋을 사용합니다.

https://huggingface.co/datasets/TheKernel01/140k-Real-and-Fake-Faces

전체 데이터셋을 다운로드하지 말고 작은 샘플만 사용하세요.

초기 실험 설정:

- Train: Real 200장 + Fake 200장
- Validation: Real 50장 + Fake 50장
- Test: Real 50장 + Fake 50장
- 총 600장

데이터셋의 공식 Train, Validation, Test 분할을 우선 활용합니다.

`datasets` 라이브러리의 Streaming 기능을 검토하여 필요한 수량만 수집하고, 전체 4GB 다운로드는 피합니다.

스트리밍이 실패하면 접근 가능한 다른 공개 Real/Fake 얼굴 데이터셋이나 작은 파일 단위 다운로드 방식으로 대체합니다.

실험 규모가 너무 작아서 결과가 불안정하면 별도의 확장 설정도 만듭니다.

- Quick: 600장
- Extended: 1,200장

기본값은 Quick으로 설정합니다.

## 3. 데이터 분할 및 공정성

중요한 요구사항입니다.

1. CNN은 Train 데이터로 학습합니다.
2. RAG 검색 DB는 동일한 Train 이미지와 라벨로 구성합니다.
3. GenD는 공개 사전학습 가중치를 사용합니다.
4. 네 모델 모두 완전히 동일한 Test 이미지로 평가합니다.
5. Validation 데이터는 CNN 학습 종료 판단, RAG 임계값, 하이브리드 결합 가중치 선택에만 활용합니다.
6. Test 정답은 모델 설정이나 임계값 결정에 절대 사용하지 않습니다.
7. 동일 파일 및 가능한 이미지 중복을 확인합니다.
8. 중복 이미지가 Train, Validation, Test에 걸치지 않도록 검사합니다.

가능하다면 얼굴 ID 또는 원본 이미지 ID 단위의 데이터 누수 여부도 확인하고, 정보가 없어 검사할 수 없는 항목은 한계로 기록합니다.

Exact Match 성능은 별도 실험으로 평가하고, 일반 Test 정확도와 섞지 않습니다.

## 4. CNN 구현

다음 기능을 추가해 주세요.

- TinyCNN 클래스 구현
- Dataset 및 DataLoader 구현
- GPU 학습 루프
- Train Loss 기록
- Validation Loss 기록
- Accuracy 측정
- Early Stopping
- 최적 Weight 저장
- 학습 결과 그래프
- 이미지 한 장을 입력받는 CNN 추론 함수

가급적 학습 시간이 짧고 코드가 이해하기 쉬운 구조를 사용합니다.

또한 학습 전에 반드시 Real/Fake 샘플 이미지와 라벨이 올바른지 시각적으로 확인합니다.

## 5. 네 모델의 성능 비교

동일한 Test 데이터에서 다음을 측정합니다.

- Accuracy
- Precision
- Recall
- F1-score
- Confusion Matrix
- 평균 추론 시간(ms/image)
- 학습 시간(CNN)
- GPU 최대 메모리 사용량

평가 시 모델을 Eval 모드로 전환하고, GPU 작업을 동기화하여 추론 시간을 측정합니다.

CNN은 학습 시간과 추론 시간을 분리해서 보고합니다.

검색 기반 방법은 임베딩 생성과 검색에 걸린 시간도 포함하도록 합니다.

## 6. 결과 시각화

다음 결과물을 작성합니다.

1. 4개 모델의 Accuracy 비교 막대그래프
2. F1-score 비교 그래프
3. 추론 시간 비교 그래프
4. CNN Train/Validation Loss 그래프
5. 네 모델의 Confusion Matrix
6. 모델 간 분류 결과가 서로 달랐던 이미지 사례 5개

그래프의 제목과 범례는 한국어 또는 영어로 일관되게 작성합니다.

결과 CSV:
`/content/deepfake_project/results/model_comparison.csv`

개별 예측 CSV:
`/content/deepfake_project/results/predictions.csv`

## 7. Gradio 웹 UI 확장

기존 Gradio UI에 CNN을 추가합니다.

이미지를 업로드하면 다음 결과를 출력합니다.

- CNN: Real/Fake
- GenD: Real/Fake
- RAG: Real/Fake
- GenD + RAG: Real/Fake

각 모델의 점수와 실행 시간도 표시하고, RAG의 Top-3 유사 이미지도 출력합니다.

## 8. 무료 GPU 환경 최적화

Google Colab Tesla T4에서 실행할 수 있도록 최적화해 주세요.

- 전체 대규모 데이터 다운로드 금지
- FP16 추론 사용 가능 여부 검토
- 작은 배치 크기
- 임베딩 결과 캐싱
- 불필요한 모델 중복 로딩 방지
- GPU 메모리 정리
- 데이터 다운로드 실패 시 복구 방법 제공
- PyTorch와 CUDA가 정상 작동하므로 무조건 재설치하지 말 것

## 9. 구현 및 실행 검증

기존 코드를 삭제하거나 처음부터 다시 작성하지 말고, 현재 구현을 확인한 뒤 필요한 부분을 추가합니다.

기존 노트북을 백업하고 CNN 비교 기능을 포함한 새 노트북을 작성해 주세요.

새 노트북 이름:
`GenD_RAG_CNN_Comparison.ipynb`

필요한 경우 별도의 Python 모듈을 만들어도 됩니다.

반드시 다음 순서로 검증합니다.

1. 기존 GenD 모델이 정상 동작하는지 확인
2. 실제 데이터셋 다운로드 확인
3. CNN 학습 실행
4. CLIP + FAISS 검색 실행
5. 하이브리드 예측 실행
6. 네 모델 공통 Test 평가
7. 그래프 및 CSV 생성
8. Gradio UI 실행

실행이 불가능한 경우 더미 결과를 사용하지 마세요.

Codex 실행 환경에서 인터넷 또는 GPU 접근이 차단된다면 Colab에서 실제 실행할 셀을 작성하고, 해당 단계가 미검증 상태임을 명확하게 보고하세요.

## 10. 최종 결과

작업이 끝나면 다음을 보고해 주세요.

- 생성하거나 수정한 파일
- 데이터셋 실제 사용 이미지 수
- CNN 학습 성공 여부
- GenD 추론 성공 여부
- RAG 검색 성공 여부
- Hybrid 실행 성공 여부
- 각 모델의 실제 Accuracy 및 F1
- 실행하지 못한 작업과 오류
- Google Colab에서 다음으로 실행할 셀

**가장 중요한 것은 복잡한 연구용 시스템이 아니라, 무료 Google Colab에서 실제로 돌아가는 작고 명확한 4개 모델 비교 실험을 완성하는 것입니다.**

In [ ]:
%%writefile /content/deepfake_project/qwen_install_prompt.md
Google Colab 환경에서 `Qwen/Qwen3-VL-2B-Instruct` 모델을 설치하고 실행할 수 있도록 설정해줘.

**요구사항**
1. Google Colab 무료 T4 GPU(16GB VRAM) 기준으로 구성해줘.
2. Hugging Face의 `Qwen/Qwen3-VL-2B-Instruct` 모델을 사용해줘.
3. GPU 메모리 절약을 위해 bitsandbytes 4-bit 양자화를 적용해줘.
4. `transformers`, `accelerate`, `bitsandbytes`, `qwen-vl-utils` 등 필요한 패키지를 설치해줘.
5. 모델과 프로세서를 불러오고 CUDA 및 GPU 메모리 사용량을 확인해줘.
6. 이미지 1장과 텍스트 프롬프트를 입력하면 모델이 답변하는 추론 함수를 구현해줘.
7. 실제 테스트 이미지를 이용해 정상적으로 추론되는지 검증해줘.
8. 패키지 버전 충돌이나 CUDA 오류가 발생하면 Colab 환경에 맞게 수정해줘.

**추가 조건**
- 별도의 유료 API 없이 무료 오픈소스 모델만 사용해줘.
- 이후 정치 홍보물 이미지의 Real/Fake 분류 및 RAG 기반 성능 비교 실험에 활용할 예정이야.
- 기존 파일은 삭제하거나 덮어쓰지 말아줘.
- 설치 코드, 모델 로딩 코드, 추론 테스트 코드를 셀별로 구분해줘.
- 우선 모델 설치와 기본 추론까지만 구현하고, RAG나 파인튜닝은 아직 구현하지 마.
- 코드만 작성하지 말고 실제 Colab에서 실행하여 정상 작동 여부까지 확인해줘.

In [ ]:
%cd /content/deepfake_project

In [ ]:
!codex exec --skip-git-repo-check --sandbox workspace-write - < /content/deepfake_project/qwen_install_prompt.md

In [ ]:
%%writefile /content/prompt.md

# 프로젝트: GenD + RAG 기반 딥페이크 이미지 분류 시스템

Google Colab 무료 GPU 환경에서 실행할 수 있는
딥페이크 이미지 분류 시스템을 구현해 줘.

# 목표
1. GenD 단독 분류
2. CLIP + FAISS 기반 RAG 이미지 검색 분류
3. GenD + RAG 하이브리드 분류
4. 세 모델의 Accuracy, F1, 추론 시간 비교
5. Gradio 기반 웹 UI 구현

# 요구사항
- PyTorch 기반
- 공개 사전학습 모델 사용
- 무료 GPU 환경에서 실행
- 데이터셋 자동 다운로드
- Real / Fake 분류
- 실제 실행 후 오류 수정
- 실행 가능한 ipynb 노트북 생성
- 상세한 한국어 주석 작성
- 동일 이미지 검색과 유사 이미지 검색 구분
- 검색 DB와 테스트 데이터 분리
- 임계값은 검증 데이터에서 결정

# 구현 순서
1. GenD 모델 로딩 및 추론
2. CLIP 임베딩 생성
3. FAISS 검색 DB 구축
4. RAG 분류 구현
5. GenD + RAG 결합
6. 성능 비교
7. Gradio 데모 제작

위 내용을 단계별로 실제 구현해 줘.

In [ ]:
!codex exec --skip-git-repo-check --sandbox workspace-write - < /content/prompt.md

In [ ]:
!nvidia-smi

In [ ]:
import torch

print("PyTorch:", torch.__version__)
print("CUDA 사용 가능:", torch.cuda.is_available())

if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))
    x = torch.randn(1000, 1000, device="cuda")
    print("GPU 연산 성공:", (x @ x).shape)

In [ ]:
%%writefile /content/deepfake_project/cnn_prompt.md
현재 `/content/deepfake_project/GenD_RAG_Colab.ipynb`에 구현된 GenD, RAG, Hybrid 시스템을 유지하면서 일반 CNN 모델을 추가해 주세요.

Google Colab의 Tesla T4 GPU와 PyTorch CUDA 연산은 정상 작동합니다.

**목표: 일반 CNN / GenD / RAG / GenD + RAG 총 4가지 방법의 실제 이미지 분류 성능 비교**

요구사항:

1. 공개 Real/Fake 얼굴 데이터셋에서 총 600장만 준비합니다.
   - Train: 400장
   - Validation: 100장
   - Test: 100장
   - 각 분할의 Real/Fake 비율 1:1
2. 일반 CNN을 PyTorch로 구현합니다.
   - Conv2d 기반 3~4개 블록
   - 이미지 크기 128×128
   - Batch Size 16
   - 학습 5 Epoch
   - Adam Optimizer
3. CNN은 Train 데이터로 학습하고 RAG는 동일한 Train 데이터를 검색 DB로 사용합니다.
4. GenD는 공식 사전학습 모델을 사용합니다.
5. 네 모델을 동일한 Test 이미지 100장으로 평가합니다.
6. Accuracy, Precision, Recall, F1, Confusion Matrix, 평균 추론 시간을 비교합니다.
7. CNN 학습 그래프와 네 모델 비교 그래프를 만듭니다.
8. 비교 결과를 CSV로 저장합니다.
9. 기존 Gradio UI에 CNN 분류 결과를 추가합니다.

**중요 조건:**
- 기존 코드는 삭제하지 않습니다.
- 새 노트북 `GenD_RAG_CNN_Comparison.ipynb`를 생성합니다.
- 실제 GenD 체크포인트와 CLIP, FAISS를 사용합니다.
- 더미 모델이나 합성 데이터의 성능을 실제 결과처럼 보고하지 않습니다.
- 데이터셋 전체 다운로드를 피하고, 필요한 소량만 가져옵니다.
- Test 데이터는 임계값 조정이나 학습에 사용하지 않습니다.
- 모든 모델이 정상 작동하는지 확인하고, 오류가 발생하면 수정합니다.
- Codex 실행 환경에서 GPU나 인터넷이 차단되면 Colab에서 실행할 수 있는 코드를 만들고 미검증 상태를 보고합니다.

최종적으로 **CNN과 GenD의 성능 차이, RAG 단독 성능, RAG를 결합했을 때의 성능 변화**를 비교할 수 있도록 구현해 주세요.

In [ ]:
%cd /content/deepfake_project

In [ ]:
!codex exec --skip-git-repo-check --sandbox workspace-write - < cnn_prompt.md

In [ ]:
!ls -lh /content/deepfake_project/*.ipynb

In [ ]:
%%writefile /content/deepfake_project/dashboard_prompt.md
# GenD + RAG + CNN 프로젝트 종합 결과 대시보드 제작

현재 `/content/deepfake_project`에 구현된 딥페이크 이미지 분류 프로젝트를 분석하고, 지금까지 사용한 기술, 구현 방법, 실험 방법 및 실제 측정 결과를 한눈에 볼 수 있는 대시보드를 만들어 주세요.

## 1. 프로젝트 구성 정리
- 사용한 라이브러리와 모델
- CNN, GenD, RAG, Hybrid의 구조
- 각 모델이 Real/Fake를 분류하는 과정
- 데이터셋 출처 및 실제 사용 수량
- Colab GPU 환경

실제 코드에 존재하는 기능과 아직 구현되지 않은 기능을 구분하세요.

## 2. 실험 과정 정리
- Train / Validation / Test 분할
- CNN 학습 설정
- GenD 사전학습 모델
- CLIP 임베딩 및 FAISS 검색 방식
- GenD + RAG 결합 규칙
- 정확도 및 추론 시간 측정 방법
- 데이터 누수 방지 방법

## 3. 실제 성능 비교
CNN, GenD, RAG, Hybrid를 동일한 Test 데이터셋에서 평가하고 다음 지표를 비교하세요.

- Accuracy
- Precision
- Recall
- F1
- 평균 추론 시간
- GPU 메모리 사용량

실제 실험 데이터만 사용하고, 기존 Smoke Test 결과는 별도로 표시하세요.

## 4. 이전과 이후 성능 비교
다음 비교를 시각화하세요.

- CNN → GenD: 사전학습 전문 모델과 소규모 CNN의 차이
- GenD → GenD + RAG: 검색 결합 전후의 정확도 및 F1 변화
- RAG 단독 → Hybrid: 이미지 검색과 모델 예측을 결합한 효과

성능 변화는 퍼센트포인트(%p)로 표시하세요.

개선되지 않았거나 오히려 악화된 경우에도 그대로 보고하세요.

## 5. 시각화
- 4개 모델 Accuracy 비교
- 4개 모델 F1 비교
- 추론 시간 비교
- CNN 학습 Loss 그래프
- Confusion Matrix
- RAG 결합으로 정답이 된 사례
- RAG 결합으로 오답이 된 사례
- GenD와 RAG 예측이 충돌한 사례

## 6. 대시보드 제작
HTML 기반의 보기 좋은 대시보드를 만들어 주세요.

구성:
1. 프로젝트 개요
2. 사용 기술 및 모델 구조
3. 데이터셋과 실험 설정
4. 네 모델 성능 비교
5. RAG 결합 전후 성능 변화
6. 이미지별 예측 비교
7. 결과 해석 및 한계

출력 파일:
`/content/deepfake_project/results/dashboard.html`

가능하다면 PDF 보고서도 생성하고, 전체 실험 설정과 결과를 CSV 및 JSON으로 저장하세요.

**중요: 실제 실행하지 않은 실험은 미실행으로 표시하고, Mock/Smoke Test 수치를 실제 성능 결과로 사용하지 마세요. 실제 평가 데이터가 없다면 해당 항목을 미측정으로 표시하고 필요한 실행 코드를 제공하세요.**

In [ ]:
%cd /content/deepfake_project
!codex exec --skip-git-repo-check --sandbox workspace-write - < dashboard_prompt.md

In [ ]:
%cd /content/deepfake_project
%run build_dashboard.py --run

In [ ]:
%pip install -q faiss-cpu

In [ ]:
%cd /content/deepfake_project
%run build_dashboard.py --run

In [ ]:
import transformers
import torch

print("Transformers:", transformers.__version__)
print("Default device:", torch.get_default_device())

In [ ]:
%pip install -q "transformers==4.56.2"

In [ ]:
%%writefile /content/deepfake_project/fix_prompt.md

# GenD 모델 로딩 오류 수정 및 실제 실험 재실행

프로젝트 경로: `/content/deepfake_project`

현재 `build_dashboard.py` 실행 중 `GenD_RAG_CNN_Comparison.ipynb`의 cell-4에서 오류가 발생했습니다.

오류:

`RuntimeError: You are using from_pretrained with a meta device context manager or torch.set_default_device('meta')`

현재 환경:
- Tesla T4 GPU 정상
- CUDA 연산 정상
- PyTorch 2.14.1+cu130
- FP16 설정
- GenD 가중치 다운로드 완료
- CLIP 관련 파일 다운로드 완료

## 오류 분석 및 수정

1. 현재 설치된 Transformers 및 Accelerate 버전을 확인합니다.
2. GenD 공식 구현의 권장 버전 `transformers==4.56.2`와 비교합니다.
3. GenD 내부 `CLIPEncoder`에서 `CLIPModel.from_pretrained()`를 호출하는 중첩 로딩 구조를 확인합니다.
4. Transformers의 meta device 초기화 과정에서 충돌하는지 분석합니다.
5. 우선 공식 권장 버전에 맞춰 호환성을 복구합니다.
6. 그래도 실패하면 GenD 체크포인트 구조를 검증한 뒤 내부 CLIP 초기화 방식을 안전하게 수정합니다.
7. 중첩된 meta device 컨텍스트 문제를 단순히 CUDA 문제로 취급하지 마세요.
8. 이미 정상 작동하는 PyTorch와 CUDA는 불필요하게 재설치하지 마세요.

## 수정 후 검증

1. 공식 GenD 체크포인트가 정상적으로 로드되는지 확인합니다.
2. 가중치가 실제로 로드되었는지 검증합니다.
3. Real/Fake 이미지 1장씩으로 실제 추론을 수행합니다.
4. 이후 CNN, GenD, RAG, Hybrid의 공통 테스트를 실행합니다.
5. 실제 성능 결과를 CSV와 HTML 대시보드에 저장합니다.

수정한 파일과 변경 내용을 알려주고, 실제로 실행한 단계와 미검증 단계를 구분해 보고하세요.

기존 프로젝트 구조와 실험 설정은 유지해 주세요.

In [ ]:
%cd /content/deepfake_project
!codex exec --skip-git-repo-check --sandbox workspace-write - < fix_prompt.md

In [ ]:
%pip uninstall -y diffusers
%pip install -q \
    "transformers==4.56.2" \
    "huggingface-hub==0.36.2" \
    "gradio==5.49.1"

In [ ]:
import torch
import transformers
import gradio
import huggingface_hub

print("PyTorch:", torch.__version__)
print("Transformers:", transformers.__version__)
print("Gradio:", gradio.__version__)
print("HF Hub:", huggingface_hub.__version__)
print("CUDA:", torch.cuda.is_available())

In [ ]:
from importlib.metadata import version

for pkg in ["transformers", "gradio", "huggingface-hub"]:
    print(pkg, version(pkg))

In [ ]:
import torch
import transformers
import gradio
import huggingface_hub

print("Transformers:", transformers.__version__)
print("Gradio:", gradio.__version__)
print("HF Hub:", huggingface_hub.__version__)
print("CUDA:", torch.cuda.is_available())

In [ ]:
import torch
import transformers
import gradio
import huggingface_hub

print("Transformers:", transformers.__version__)
print("Gradio:", gradio.__version__)
print("HF Hub:", huggingface_hub.__version__)
print("CUDA:", torch.cuda.is_available())

In [ ]:
%cd /content/deepfake_project
!python build_dashboard.py --run

In [ ]:
!find /content/deepfake_project/results \
  -maxdepth 3 -type f | sort

In [ ]:
from pathlib import Path
from IPython.display import HTML, display

files = list(Path("/content/deepfake_project/results").rglob("*.html"))

for f in files:
    print(f)

if files:
    display(HTML(files[-1].read_text(encoding="utf-8")))

In [ ]:
from google.colab import files
import shutil

shutil.make_archive(
    "/content/deepfake_results",
    "zip",
    "/content/deepfake_project/results"
)

files.download("/content/deepfake_results.zip")

In [ ]:
%pip install -U "transformers>=5,<6" "huggingface-hub>=1.32,<2" accelerate bitsandbytes qwen-vl-utils

In [ ]:
import torch
import transformers
import huggingface_hub
import bitsandbytes

print("PyTorch:", torch.__version__)
print("Transformers:", transformers.__version__)
print("HuggingFace Hub:", huggingface_hub.__version__)
print("BitsAndBytes:", bitsandbytes.__version__)
print("CUDA:", torch.cuda.is_available())

In [ ]:
!pip check

In [ ]:
from transformers import (
    Qwen3VLForConditionalGeneration,
    AutoProcessor,
    BitsAndBytesConfig
)

MODEL_ID = "Qwen/Qwen3-VL-2B-Instruct"

quant_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.float16,
    bnb_4bit_use_double_quant=True
)

model = Qwen3VLForConditionalGeneration.from_pretrained(
    MODEL_ID,
    quantization_config=quant_config,
    dtype=torch.float16,
    device_map="auto",
    attn_implementation="sdpa"
)

processor = AutoProcessor.from_pretrained(MODEL_ID)
model.eval()

print("모델 로드 완료")

In [ ]:
%pip install -q faiss-cpu

In [ ]:
!git clone https://github.com/yermandy/GenD.git /content/GenD

In [ ]:
import torch
import faiss
import numpy as np

from PIL import Image
from transformers import CLIPVisionModelWithProjection, CLIPImageProcessor

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"

CLIP_ID = "openai/clip-vit-base-patch32"

clip_model = CLIPVisionModelWithProjection.from_pretrained(
    CLIP_ID
).to(DEVICE).eval()

clip_processor = CLIPImageProcessor.from_pretrained(CLIP_ID)

print("CLIP 모델 로드 완료")

In [ ]:
!git clone https://github.com/Grow22/DeepLearning_6team.git /content/DeepLearning_6team

%cd /content/DeepLearning_6team

%pip install -q huggingface_hub pyarrow pillow

!python data_preparation/prepare_all.py

In [ ]:
import torch
import numpy as np
import pandas as pd
import transformers
import os

from PIL import Image
from pathlib import Path

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"

print("PyTorch:", torch.__version__)
print("Transformers:", transformers.__version__)
print("Device:", DEVICE)

if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))
    print("GPU 사용량:", round(torch.cuda.memory_allocated() / 1024**3, 2), "GB")

print("Qwen 로드 여부:", "model" in globals())
print("CLIP 로드 여부:", "clip_model" in globals())
print("CLIP Processor:", "clip_processor" in globals())

DATA_ROOT = Path("/content/DeepLearning_6team/data/ai_photos")

print("데이터셋 존재:", DATA_ROOT.exists())
print("CSV 존재:", (DATA_ROOT / "ai_photos.csv").exists())

In [ ]:
import torch

from transformers import (
    Qwen3VLForConditionalGeneration,
    AutoProcessor,
    BitsAndBytesConfig
)

MODEL_ID = "Qwen/Qwen3-VL-2B-Instruct"

quant_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.float16,
    bnb_4bit_use_double_quant=True
)

model = Qwen3VLForConditionalGeneration.from_pretrained(
    MODEL_ID,
    quantization_config=quant_config,
    dtype=torch.float16,
    device_map="auto",
    attn_implementation="sdpa"
)

processor = AutoProcessor.from_pretrained(MODEL_ID)

model.eval()

print("Qwen 모델 로드 완료")
print("GPU 사용량:", round(torch.cuda.memory_allocated() / 1024**3, 2), "GB")

In [ ]:
@torch.inference_mode()
def baseline_predict(image_path):
    image = Image.open(image_path).convert("RGB")

    messages = [{
        "role": "user",
        "content": [
            {"type": "image", "image": image},
            {
                "type": "text",
                "text": (
                    "Determine whether this image is a real photograph "
                    "or an AI-generated image. "
                    "Answer only REAL or FAKE."
                )
            }
        ]
    }]

    text = processor.apply_chat_template(
        messages,
        tokenize=False,
        add_generation_prompt=True
    )

    inputs = processor(
        text=[text],
        images=[image],
        return_tensors="pt"
    ).to(model.device)

    outputs = model.generate(
        **inputs,
        max_new_tokens=16,
        do_sample=False
    )

    result = processor.decode(
        outputs[0][inputs.input_ids.shape[1]:],
        skip_special_tokens=True
    ).strip().upper()

    return {
        "prediction": result if result in ["REAL", "FAKE"] else "UNKNOWN",
        "raw_output": result
    }

print("Baseline 탐지 함수 생성 완료")

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt

df = pd.read_csv(DATA_ROOT / "ai_photos.csv")

# Train 데이터에서 테스트용 이미지 선택
samples = (
    df[(df["split"] == "train") & df["normalized_path"].notna()]
    .groupby("label")
    .sample(n=1, random_state=42)
)

for row in samples.itertuples():
    image_path = DATA_ROOT / row.normalized_path
    ground_truth = "REAL" if row.label == 0 else "FAKE"

    result = baseline_predict(image_path)

    plt.figure(figsize=(4, 4))
    plt.imshow(Image.open(image_path))
    plt.axis("off")
    plt.show()

    print("정답:", ground_truth)
    print("Qwen 판정:", result["prediction"])
    print("모델 원본 응답:", result["raw_output"])
    print("-" * 40)

In [ ]:
import torch
import numpy as np

from transformers import (
    CLIPVisionModelWithProjection,
    CLIPImageProcessor
)

CLIP_ID = "openai/clip-vit-base-patch32"

clip_model = CLIPVisionModelWithProjection.from_pretrained(
    CLIP_ID,
    dtype=torch.float16
).to(DEVICE).eval()

clip_processor = CLIPImageProcessor.from_pretrained(CLIP_ID)

print("CLIP 모델 로드 완료")
print("임베딩 차원:", clip_model.config.projection_dim)
print("전체 GPU 사용량:",
      round(torch.cuda.memory_allocated() / 1024**3, 2), "GB")

In [ ]:
import torch.nn.functional as F

@torch.inference_mode()
def get_embedding(image_path):
    with Image.open(image_path) as img:
        image = img.convert("RGB")

    inputs = clip_processor(
        images=image,
        return_tensors="pt"
    )

    pixel_values = inputs["pixel_values"].to(
        device=DEVICE,
        dtype=clip_model.dtype
    )

    outputs = clip_model(pixel_values=pixel_values)

    embedding = F.normalize(
        outputs.image_embeds.float(),
        p=2,
        dim=-1
    )

    return embedding.cpu().numpy().astype("float32")


# 임베딩 테스트
sample_path = DATA_ROOT / samples.iloc[0]["normalized_path"]

embedding = get_embedding(sample_path)

print("임베딩 크기:", embedding.shape)
print("벡터 Norm:", np.linalg.norm(embedding))
print("데이터 타입:", embedding.dtype)
print("임베딩 일부:", embedding[0][:10])

In [ ]:
%pip install -q faiss-cpu

In [ ]:
import faiss

df = pd.read_csv(DATA_ROOT / "ai_photos.csv")

# 정규화 이미지가 존재하는 데이터만 사용
df = df.dropna(subset=["normalized_path"]).copy()

df["path"] = df["normalized_path"].apply(
    lambda p: str(DATA_ROOT / p)
)

train_df = df[df["split"] == "train"].copy()
val_df = df[df["split"] == "validation"].copy()
test_df = df[df["split"].str.startswith("test")].copy()

# FAISS 인덱스 생성
index = faiss.IndexFlatIP(512)

rag_labels = []
rag_paths = []

print("Train:", len(train_df))
print("Validation:", len(val_df))
print("Test:", len(test_df))
print("FAISS 초기화 완료")
print("현재 DB 크기:", index.ntotal)

display(df.groupby(["split", "label"]).size().unstack())

In [ ]:
from tqdm.auto import tqdm
import torch.nn.functional as F

BATCH_SIZE = 32

@torch.inference_mode()
def get_batch_embeddings(image_paths):
    images = []

    for path in image_paths:
        with Image.open(path) as img:
            images.append(img.convert("RGB"))

    inputs = clip_processor(
        images=images,
        return_tensors="pt"
    )

    pixel_values = inputs["pixel_values"].to(
        device=DEVICE,
        dtype=clip_model.dtype
    )

    outputs = clip_model(pixel_values=pixel_values)

    embeddings = F.normalize(
        outputs.image_embeds.float(),
        p=2,
        dim=-1
    )

    return embeddings.cpu().numpy().astype("float32")


# 기존 DB 초기화
index.reset()
rag_labels.clear()
rag_paths.clear()

# Train 이미지 임베딩 및 저장
for start in tqdm(
    range(0, len(train_df), BATCH_SIZE),
    desc="RAG DB 구축"
):
    batch = train_df.iloc[start:start + BATCH_SIZE]

    paths = batch["path"].tolist()
    embeddings = get_batch_embeddings(paths)

    index.add(embeddings)

    rag_paths.extend(paths)
    rag_labels.extend(
        ["FAKE" if label == 1 else "REAL"
         for label in batch["label"]]
    )

assert index.ntotal == len(rag_labels) == len(rag_paths)

print("RAG DB 구축 완료")
print("전체 이미지:", index.ntotal)
print("REAL:", rag_labels.count("REAL"))
print("FAKE:", rag_labels.count("FAKE"))
print("벡터 차원:", index.d)

In [ ]:
def retrieve_images(image_path, k=5):
    embedding = get_embedding(image_path)

    scores, indices = index.search(embedding, k)

    results = []

    for score, idx in zip(scores[0], indices[0]):
        results.append({
            "path": rag_paths[idx],
            "label": rag_labels[idx],
            "similarity": float(score)
        })

    return results


# Validation 이미지 한 장 선택
sample = val_df.sample(n=1, random_state=42).iloc[0]

neighbors = retrieve_images(sample["path"])

# 검색 결과 시각화
fig, axes = plt.subplots(1, 6, figsize=(18, 4))

axes[0].imshow(Image.open(sample["path"]))
axes[0].set_title(
    "Query\n" + ("REAL" if sample["label"] == 0 else "FAKE")
)
axes[0].axis("off")

for i, item in enumerate(neighbors):
    axes[i + 1].imshow(Image.open(item["path"]))
    axes[i + 1].set_title(
        f'Top {i+1}: {item["label"]}\n'
        f'Sim: {item["similarity"]:.4f}'
    )
    axes[i + 1].axis("off")

plt.tight_layout()
plt.show()

display(pd.DataFrame(neighbors))

In [ ]:
SIMILARITY_THRESHOLD = 0.95
AGREEMENT_THRESHOLD = 0.8

def rag_predict(image_path, k=5):
    neighbors = retrieve_images(image_path, k)

    labels = [n["label"] for n in neighbors]

    real_count = labels.count("REAL")
    fake_count = labels.count("FAKE")

    similarity = neighbors[0]["similarity"]
    agreement = max(real_count, fake_count) / k

    if (
        similarity >= SIMILARITY_THRESHOLD
        and agreement >= AGREEMENT_THRESHOLD
        and real_count != fake_count
    ):
        prediction = "REAL" if real_count > fake_count else "FAKE"
        route = "RAG"
    else:
        prediction = "UNKNOWN"
        route = "GenD"

    return {
        "prediction": prediction,
        "similarity": similarity,
        "agreement": agreement,
        "real_count": real_count,
        "fake_count": fake_count,
        "route": route
    }


result = rag_predict(sample["path"])

for key, value in result.items():
    print(f"{key}: {value}")

In [ ]:
import sys
import torch

from huggingface_hub import hf_hub_download
from safetensors.torch import load_file

sys.path.insert(0, "/content/GenD")

from src.hf.modeling_gend import GenD, GenDConfig

GEND_ID = "yermandy/GenD_CLIP_L_14"

# 1. 모델 설정 불러오기
gend_config = GenDConfig.from_pretrained(GEND_ID)

# 2. meta device 초기화를 우회하여 모델 생성
with torch.device("cpu"):
    gend_model = GenD(gend_config)

# 3. 공식 사전 학습 가중치 불러오기
weights_path = hf_hub_download(
    repo_id=GEND_ID,
    filename="model.safetensors"
)

state_dict = load_file(weights_path, device="cpu")

# 4. 모델에 가중치 적용
gend_model.load_state_dict(state_dict, strict=True)

# 5. GPU 이동
gend_model = gend_model.to(DEVICE).eval()

del state_dict

print("GenD 모델 로드 완료")
print("GPU 사용량:",
      round(torch.cuda.memory_allocated() / 1024**3, 2), "GB")
print("Qwen 유지:", "model" in globals())
print("CLIP 유지:", "clip_model" in globals())
print("RAG DB 크기:", index.ntotal)

In [ ]:
@torch.inference_mode()
def gend_predict(image_path):
    with Image.open(image_path) as img:
        image = img.convert("RGB")

    tensor = gend_model.feature_extractor.preprocess(
        image
    ).unsqueeze(0).to(DEVICE)

    logits = gend_model(tensor)

    probs = torch.softmax(logits.float(), dim=-1)

    real_score = probs[0, 0].item()
    fake_score = probs[0, 1].item()

    return {
        "prediction": "FAKE" if fake_score >= 0.5 else "REAL",
        "real_score": real_score,
        "fake_score": fake_score
    }


# 기존 Qwen 테스트에 사용했던 이미지로 비교
for row in samples.itertuples():
    image_path = DATA_ROOT / row.normalized_path
    ground_truth = "REAL" if row.label == 0 else "FAKE"

    result = gend_predict(image_path)

    plt.figure(figsize=(4, 4))
    plt.imshow(Image.open(image_path))
    plt.axis("off")
    plt.show()

    print("정답:", ground_truth)
    print("GenD 판정:", result["prediction"])
    print(f"REAL 점수: {result['real_score']:.4f}")
    print(f"FAKE 점수: {result['fake_score']:.4f}")
    print("-" * 40)

In [ ]:
def hybrid_predict(image_path):

    # 1차: RAG 판정
    rag = rag_predict(image_path)

    if rag["prediction"] != "UNKNOWN":
        return {
            "prediction": rag["prediction"],
            "stage": "RAG",
            "similarity": rag["similarity"],
            "agreement": rag["agreement"],
            "fake_score": None
        }

    # 2차: GenD 판정
    gend = gend_predict(image_path)

    return {
        "prediction": gend["prediction"],
        "stage": "GenD",
        "similarity": rag["similarity"],
        "agreement": rag["agreement"],
        "fake_score": gend["fake_score"]
    }


# Validation 이미지로 통합 테스트
result = hybrid_predict(sample["path"])

print("정답:", "REAL" if sample["label"] == 0 else "FAKE")

for key, value in result.items():
    print(f"{key}: {value}")

In [ ]:
from tqdm.auto import tqdm
import pandas as pd

validation_results = []

for row in tqdm(
    val_df.itertuples(),
    total=len(val_df),
    desc="Validation 평가"
):
    rag = rag_predict(row.path)
    gend = gend_predict(row.path)

    validation_results.append({
        "image_id": row.image_id,
        "label": row.label,
        "similarity": rag["similarity"],
        "agreement": rag["agreement"],
        "real_count": rag["real_count"],
        "fake_count": rag["fake_count"],
        "gend_fake_score": gend["fake_score"]
    })

val_results = pd.DataFrame(validation_results)

val_results.to_csv(
    "/content/validation_results.csv",
    index=False
)

print("Validation 추론 완료:", len(val_results))

display(val_results.head())

display(
    val_results.groupby("label")[
        ["similarity", "agreement", "gend_fake_score"]
    ].describe()
)

In [ ]:
import numpy as np
import pandas as pd

from sklearn.metrics import roc_auc_score

y_true = val_results["label"].to_numpy()
similarity = val_results["similarity"].to_numpy()
agreement = val_results["agreement"].to_numpy()
gend_scores = val_results["gend_fake_score"].to_numpy()

rag_preds = (
    val_results["fake_count"] > val_results["real_count"]
).astype(int).to_numpy()

search_results = []

# Grid Search
for sim_t in np.r_[0.0, np.arange(0.60, 1.01, 0.01), 1.01]:
    for agree_t in [0.6, 0.8, 1.0]:
        for gend_t in np.arange(0.0, 1.01, 0.01):

            use_rag = (
                (similarity >= sim_t)
                & (agreement >= agree_t)
            )

            gend_preds = (gend_scores >= gend_t).astype(int)

            final_preds = np.where(
                use_rag,
                rag_preds,
                gend_preds
            )

            accuracy = np.mean(final_preds == y_true)

            search_results.append({
                "similarity_threshold": sim_t,
                "agreement_threshold": agree_t,
                "gend_threshold": gend_t,
                "accuracy": accuracy,
                "rag_ratio": use_rag.mean(),
                "gend_ratio": (~use_rag).mean()
            })

grid_results = pd.DataFrame(search_results)

grid_results = grid_results.sort_values(
    ["accuracy", "rag_ratio"],
    ascending=[False, False]
).reset_index(drop=True)

best = grid_results.iloc[0]

print("===== 최적 임계값 =====")
print(f"RAG 유사도: {best['similarity_threshold']:.2f}")
print(f"RAG 일치율: {best['agreement_threshold']:.2f}")
print(f"GenD 임계값: {best['gend_threshold']:.2f}")

print("\n===== Validation 성능 =====")
print(f"Hybrid Accuracy: {best['accuracy']:.4f}")
print(f"RAG 처리 비율: {best['rag_ratio']:.2%}")
print(f"GenD 처리 비율: {best['gend_ratio']:.2%}")

print("\n===== 개별 모델 비교 =====")
print("RAG 단독 Accuracy:", np.mean(rag_preds == y_true))
print("GenD 기본 Accuracy:", np.mean((gend_scores >= 0.5) == y_true))
print("GenD 최적 Accuracy:",
      grid_results.loc[grid_results["rag_ratio"] == 0, "accuracy"].max())
print("GenD AUC:", roc_auc_score(y_true, gend_scores))

print("\n===== 상위 10개 조합 =====")
display(grid_results.head(10))

grid_results.to_csv("/content/grid_search_results.csv", index=False)

In [ ]:
SIMILARITY_THRESHOLD = float(best["similarity_threshold"])
AGREEMENT_THRESHOLD = float(best["agreement_threshold"])
GEND_THRESHOLD = float(best["gend_threshold"])

def hybrid_predict(image_path):

    rag = rag_predict(image_path)

    if rag["prediction"] != "UNKNOWN":
        return {
            "prediction": rag["prediction"],
            "stage": "RAG",
            "similarity": rag["similarity"],
            "agreement": rag["agreement"],
            "fake_score": None
        }

    gend = gend_predict(image_path)

    fake_score = gend["fake_score"]

    prediction = (
        "FAKE" if fake_score >= GEND_THRESHOLD else "REAL"
    )

    return {
        "prediction": prediction,
        "stage": "GenD",
        "similarity": rag["similarity"],
        "agreement": rag["agreement"],
        "fake_score": fake_score
    }

print("최적 임계값 적용 완료")
print("RAG 유사도:", round(SIMILARITY_THRESHOLD, 2))
print("RAG 일치율:", round(AGREEMENT_THRESHOLD, 2))
print("GenD 임계값:", round(GEND_THRESHOLD, 2))

In [ ]:
import time
from tqdm.auto import tqdm

test_results = []

for row in tqdm(
    test_df.itertuples(),
    total=len(test_df),
    desc="최종 Test 평가"
):

    ground_truth = "FAKE" if row.label == 1 else "REAL"

    # 일반 Qwen
    torch.cuda.synchronize()
    start = time.perf_counter()

    baseline = baseline_predict(row.path)

    torch.cuda.synchronize()
    baseline_time = time.perf_counter() - start

    # RAG + GenD
    torch.cuda.synchronize()
    start = time.perf_counter()

    hybrid = hybrid_predict(row.path)

    torch.cuda.synchronize()
    hybrid_time = time.perf_counter() - start

    test_results.append({
        "image_id": row.image_id,
        "split": row.split,
        "ground_truth": ground_truth,

        "baseline_prediction": baseline["prediction"],
        "baseline_raw_output": baseline["raw_output"],
        "baseline_time": baseline_time,

        "hybrid_prediction": hybrid["prediction"],
        "hybrid_stage": hybrid["stage"],
        "hybrid_time": hybrid_time,

        "similarity": hybrid["similarity"],
        "agreement": hybrid["agreement"],
        "gend_fake_score": hybrid["fake_score"]
    })

    # 중간 결과 저장
    if len(test_results) % 25 == 0:
        pd.DataFrame(test_results).to_csv(
            "/content/test_results.csv",
            index=False
        )

test_results_df = pd.DataFrame(test_results)

test_results_df.to_csv(
    "/content/test_results.csv",
    index=False
)

print("최종 Test 평가 완료:", len(test_results_df))

In [ ]:
cross_df = pd.read_csv(DATA_ROOT / "cross_test.csv")

print("전체 데이터:", len(cross_df))
print("컬럼:", cross_df.columns.tolist())

display(cross_df.head())

display(
    cross_df.groupby(["text_fake", "is_ai"]).size()
)

In [ ]:
cross_df = pd.read_csv(DATA_ROOT / "cross_test.csv")

cross_df["path"] = cross_df["normalized_path"].apply(
    lambda p: str(DATA_ROOT / p)
)

cross_df["text_truth"] = (cross_df["text_fake"] == 0)
cross_df["image_truth"] = (cross_df["is_ai"] == 0)

cross_df["ground_truth"] = (
    cross_df["text_truth"] & cross_df["image_truth"]
)

print("전체 데이터:", len(cross_df))
print("고유 이미지:", cross_df["path"].nunique())
print("고유 텍스트:", cross_df["text"].nunique())

display(cross_df["ground_truth"].value_counts())

In [ ]:
@torch.inference_mode()
def text_baseline_predict(claim):

    messages = [{
        "role": "user",
        "content": [{
            "type": "text",
            "text": (
                "Determine whether the following news claim "
                "is factually true or false.\n"
                "Answer only TRUE or FALSE.\n\n"
                f"News claim: {claim}"
            )
        }]
    }]

    text = processor.apply_chat_template(
        messages,
        tokenize=False,
        add_generation_prompt=True
    )

    inputs = processor(
        text=[text],
        return_tensors="pt"
    ).to(model.device)

    outputs = model.generate(
        **inputs,
        max_new_tokens=16,
        do_sample=False
    )

    result = processor.decode(
        outputs[0][inputs.input_ids.shape[1]:],
        skip_special_tokens=True
    ).strip().upper().rstrip(".")

    return {
        "prediction": result if result in ["TRUE", "FALSE"] else "UNKNOWN",
        "raw_output": result
    }


print("텍스트 판정 함수 생성 완료")

print(text_baseline_predict("The Earth orbits the Sun."))
print(text_baseline_predict("The Moon is larger than the Sun."))

In [ ]:
def final_decision(text_pred, image_pred):

    if text_pred == "FALSE" or image_pred == "FAKE":
        return "FALSE"

    if text_pred == "TRUE" and image_pred == "REAL":
        return "TRUE"

    return "UNKNOWN"


def baseline_total_predict(claim, image_path):

    text_result = text_baseline_predict(claim)
    image_result = baseline_predict(image_path)

    return {
        "text_prediction": text_result["prediction"],
        "image_prediction": image_result["prediction"],
        "final_prediction": final_decision(
            text_result["prediction"],
            image_result["prediction"]
        )
    }


def hybrid_total_predict(claim, image_path):

    text_result = text_baseline_predict(claim)
    image_result = hybrid_predict(image_path)

    return {
        "text_prediction": text_result["prediction"],
        "image_prediction": image_result["prediction"],
        "final_prediction": final_decision(
            text_result["prediction"],
            image_result["prediction"]
        ),
        "image_stage": image_result["stage"]
    }


# AND 판정 로직 검증
for text_pred in ["TRUE", "FALSE"]:
    for image_pred in ["REAL", "FAKE"]:
        print(
            text_pred, "+", image_pred,
            "=>", final_decision(text_pred, image_pred)
        )

In [ ]:
from tqdm.auto import tqdm
import pandas as pd

# Validation에서 결정한 임계값 고정
SIMILARITY_THRESHOLD = 0.64
AGREEMENT_THRESHOLD = 0.60
GEND_THRESHOLD = 0.28


def hybrid_predict(image_path):
    rag = rag_predict(image_path)

    if rag["prediction"] != "UNKNOWN":
        return {
            "prediction": rag["prediction"],
            "stage": "RAG"
        }

    gend = gend_predict(image_path)

    return {
        "prediction": (
            "FAKE" if gend["fake_score"] >= GEND_THRESHOLD
            else "REAL"
        ),
        "stage": "GenD"
    }


# 중복 추론 방지용 캐시
text_cache = {}
baseline_image_cache = {}
hybrid_image_cache = {}

test_results = []

for row in tqdm(
    cross_df.itertuples(),
    total=len(cross_df),
    desc="최종 통합 평가"
):

    # 1. Text 판정 (두 버전 공통)
    if row.text not in text_cache:
        text_cache[row.text] = text_baseline_predict(
            row.text
        )["prediction"]

    text_pred = text_cache[row.text]

    # 2. Baseline 이미지 판정
    if row.path not in baseline_image_cache:
        baseline_image_cache[row.path] = baseline_predict(
            row.path
        )["prediction"]

    baseline_img = baseline_image_cache[row.path]

    # 3. Hybrid 이미지 판정
    if row.path not in hybrid_image_cache:
        hybrid_image_cache[row.path] = hybrid_predict(
            row.path
        )

    hybrid = hybrid_image_cache[row.path]
    hybrid_img = hybrid["prediction"]

    # 4. AND 최종 판정
    baseline_final = final_decision(text_pred, baseline_img)
    hybrid_final = final_decision(text_pred, hybrid_img)

    test_results.append({
        "item_id": row.item_id,
        "split": row.split,

        "text_truth": "FALSE" if row.text_fake else "TRUE",
        "image_truth": "FAKE" if row.is_ai else "REAL",
        "final_truth": "TRUE" if row.ground_truth else "FALSE",

        "text_prediction": text_pred,
        "baseline_image": baseline_img,
        "hybrid_image": hybrid_img,

        "baseline_final": baseline_final,
        "hybrid_final": hybrid_final,
        "hybrid_stage": hybrid["stage"]
    })

    # 중간 결과 저장
    if len(test_results) % 50 == 0:
        pd.DataFrame(test_results).to_csv(
            "/content/cross_test_results.csv",
            index=False
        )

results_df = pd.DataFrame(test_results)

results_df.to_csv(
    "/content/cross_test_results.csv",
    index=False
)

print("전체 평가 완료:", len(results_df))
print("텍스트 추론 횟수:", len(text_cache))
print("Qwen 이미지 추론 횟수:", len(baseline_image_cache))
print("Hybrid 이미지 추론 횟수:", len(hybrid_image_cache))

display(results_df.head())

In [ ]:
import pandas as pd
import numpy as np
from sklearn.metrics import precision_score, recall_score, f1_score

def evaluate(y_true, y_pred, positive, negative):
    y_true = np.asarray(y_true)
    y_pred = np.asarray(y_pred)

    positive_recall = np.mean(
        y_pred[y_true == positive] == positive
    )
    negative_recall = np.mean(
        y_pred[y_true == negative] == negative
    )

    return {
        "Accuracy": np.mean(y_true == y_pred),
        "Precision": precision_score(
            y_true, y_pred, pos_label=positive,
            labels=[positive], average=None,
            zero_division=0
        )[0],
        "Recall": positive_recall,
        "F1-score": f1_score(
            y_true, y_pred, labels=[positive],
            average=None, zero_division=0
        )[0],
        "Balanced Accuracy": (
            positive_recall + negative_recall
        ) / 2,
        "Unknown Rate": np.mean(y_pred == "UNKNOWN")
    }


# 1. 전체 성능 비교
summary = pd.DataFrame({
    "Text (공통 Qwen)": evaluate(
        results_df["text_truth"],
        results_df["text_prediction"],
        "FALSE", "TRUE"
    ),
    "Image (Baseline)": evaluate(
        results_df["image_truth"],
        results_df["baseline_image"],
        "FAKE", "REAL"
    ),
    "Image (Hybrid)": evaluate(
        results_df["image_truth"],
        results_df["hybrid_image"],
        "FAKE", "REAL"
    ),
    "Final (Baseline)": evaluate(
        results_df["final_truth"],
        results_df["baseline_final"],
        "FALSE", "TRUE"
    ),
    "Final (Hybrid)": evaluate(
        results_df["final_truth"],
        results_df["hybrid_final"],
        "FALSE", "TRUE"
    )
}).T

print("===== 전체 성능 비교 (%) =====")
display((summary * 100).round(2))


# 2. 정확도 계산
results_df["text_correct"] = (
    results_df["text_truth"] == results_df["text_prediction"]
)

for version in ["baseline", "hybrid"]:
    results_df[f"{version}_image_correct"] = (
        results_df["image_truth"] == results_df[f"{version}_image"]
    )

    results_df[f"{version}_final_correct"] = (
        results_df["final_truth"] == results_df[f"{version}_final"]
    )


# 3. 네 가지 경우별 최종 정확도
case_results = results_df.groupby(
    ["text_truth", "image_truth"]
)[[
    "baseline_final_correct",
    "hybrid_final_correct"
]].mean() * 100

case_results["Improvement (%p)"] = (
    case_results["hybrid_final_correct"]
    - case_results["baseline_final_correct"]
)

print("===== 네 가지 경우별 정확도 (%) =====")
display(case_results.round(2))


# 4. 데이터셋별 정확도
split_results = results_df.groupby("split")[[
    "text_correct",
    "baseline_image_correct",
    "hybrid_image_correct",
    "baseline_final_correct",
    "hybrid_final_correct"
]].mean() * 100

print("===== 데이터셋별 정확도 (%) =====")
display(split_results.round(2))


# 5. RAG / GenD 사용 비율
print("===== Hybrid 처리 비율 =====")
display(
    results_df["hybrid_stage"]
    .value_counts(normalize=True)
    .mul(100).round(2)
)


# 6. Confusion Matrix
for version in ["baseline", "hybrid"]:
    print(f"===== {version.upper()} Confusion Matrix =====")

    display(pd.crosstab(
        results_df["final_truth"],
        results_df[f"{version}_final"],
        rownames=["Ground Truth"],
        colnames=["Prediction"]
    ))

In [ ]:
import json
import pandas as pd

from huggingface_hub import HfApi, hf_hub_download

DATASET_ID = "anson-huang/mirage-news"

# 팀 데이터셋과 동일한 버전 사용
meta_path = DATA_ROOT.parent.parent / (
    "data_preparation/ai_photos/cross_test_pairs.meta.json"
)

revision = json.loads(
    meta_path.read_text()
)["revision"]

repo_files = HfApi().list_repo_files(
    DATASET_ID,
    repo_type="dataset",
    revision=revision
)

def load_text_split(split):
    files = sorted([
        f for f in repo_files
        if f.startswith(f"data/{split}-")
        and f.endswith(".parquet")
    ])

    frames = []

    for filename in files:
        path = hf_hub_download(
            repo_id=DATASET_ID,
            filename=filename,
            repo_type="dataset",
            revision=revision
        )

        frames.append(
            pd.read_parquet(
                path,
                columns=["text", "label"]
            )
        )

    result = pd.concat(frames, ignore_index=True)
    result = result.dropna(subset=["text", "label"])

    result["truth"] = result["label"].map({
        0: "TRUE",
        1: "FALSE"
    })

    return result


text_train_df = load_text_split("train")
text_val_df = load_text_split("validation")

print("Text Train:", len(text_train_df))
print("Text Validation:", len(text_val_df))

display(text_train_df.head())
display(text_train_df["truth"].value_counts())

In [ ]:
%pip install -q sentence-transformers

In [ ]:
from sentence_transformers import SentenceTransformer

TEXT_EMBEDDING_ID = "sentence-transformers/all-MiniLM-L6-v2"

text_encoder = SentenceTransformer(
    TEXT_EMBEDDING_ID,
    device=DEVICE
)

# 임베딩 테스트
text_embedding = text_encoder.encode(
    ["The Earth orbits the Sun."],
    normalize_embeddings=True
)

print("Text Embedding 모델 로드 완료")
print("임베딩 차원:", text_embedding.shape)
print("벡터 Norm:", np.linalg.norm(text_embedding))
print("GPU 사용량:",
      round(torch.cuda.memory_allocated() / 1024**3, 2), "GB")

In [ ]:
import faiss
import numpy as np

# 검색할 텍스트와 라벨 저장
rag_texts = text_train_df["text"].tolist()
rag_text_labels = text_train_df["truth"].tolist()

# 배치 단위 텍스트 임베딩
text_embeddings = text_encoder.encode(
    rag_texts,
    batch_size=128,
    show_progress_bar=True,
    normalize_embeddings=True,
    convert_to_numpy=True
).astype("float32")

# 텍스트 전용 FAISS 인덱스 생성
text_index = faiss.IndexFlatIP(384)
text_index.add(text_embeddings)

assert text_index.ntotal == len(rag_texts) == len(rag_text_labels)

print("Text RAG DB 구축 완료")
print("전체 문장:", text_index.ntotal)
print("TRUE:", rag_text_labels.count("TRUE"))
print("FALSE:", rag_text_labels.count("FALSE"))
print("임베딩 크기:", text_embeddings.shape)

In [ ]:
def retrieve_texts(claim, k=5):

    embedding = text_encoder.encode(
        [claim],
        normalize_embeddings=True,
        convert_to_numpy=True
    ).astype("float32")

    scores, indices = text_index.search(embedding, k)

    results = []

    for score, idx in zip(scores[0], indices[0]):
        results.append({
            "text": rag_texts[idx],
            "label": rag_text_labels[idx],
            "similarity": float(score)
        })

    return results


# Validation 문장으로 검색 테스트
text_sample = text_val_df.sample(
    n=1, random_state=42
).iloc[0]

claim = text_sample["text"]

print("===== 입력 뉴스 =====")
print(claim)
print("정답:", text_sample["truth"])

print("\n===== 유사 뉴스 검색 결과 =====")

text_neighbors = retrieve_texts(claim)

for i, item in enumerate(text_neighbors):
    print(f"\nTop {i+1}")
    print("뉴스:", item["text"])
    print("라벨:", item["label"])
    print(f"유사도: {item['similarity']:.4f}")

In [ ]:
@torch.inference_mode()
def text_rag_predict(claim, k=5):

    # 1. 유사 뉴스 검색
    neighbors = retrieve_texts(claim, k=k)

    # 2. 검색 결과를 프롬프트로 구성
    context = "\n\n".join([
        f"Reference {i+1}:\n"
        f"Text: {item['text']}\n"
        f"Label: {item['label']}\n"
        f"Similarity: {item['similarity']:.4f}"
        for i, item in enumerate(neighbors)
    ])

    prompt = f"""
Determine whether the target news claim is TRUE or FALSE.

Use the retrieved labeled examples as references.

Instructions:
- Compare the factual details of the claims.
- Pay attention to changes in people, places, actions,
  dates, and exaggerated descriptions.
- High semantic similarity does not guarantee identical truth.
- Do not simply follow the majority label.
- If references are unrelated, use your own knowledge.

Retrieved examples:
{context}

Target news claim:
{claim}

Answer only TRUE or FALSE.
"""

    # 3. Qwen 입력 구성
    messages = [{
        "role": "user",
        "content": [{
            "type": "text",
            "text": prompt
        }]
    }]

    text = processor.apply_chat_template(
        messages,
        tokenize=False,
        add_generation_prompt=True
    )

    inputs = processor(
        text=[text],
        return_tensors="pt"
    ).to(model.device)

    # 4. Qwen 판정
    outputs = model.generate(
        **inputs,
        max_new_tokens=16,
        do_sample=False
    )

    result = processor.decode(
        outputs[0][inputs.input_ids.shape[1]:],
        skip_special_tokens=True
    ).strip().upper().rstrip(".")

    return {
        "prediction": (
            result if result in ["TRUE", "FALSE"]
            else "UNKNOWN"
        ),
        "raw_output": result,
        "neighbors": neighbors
    }


# 기존 Validation 문장으로 비교
print("정답:", text_sample["truth"])

print("일반 Qwen:",
      text_baseline_predict(claim)["prediction"])

print("Text RAG + Qwen:",
      text_rag_predict(claim)["prediction"])

In [ ]:
from tqdm.auto import tqdm
from sklearn.metrics import f1_score

text_validation_results = []
val_text_cache = {}

for row in tqdm(
    text_val_df.itertuples(),
    total=len(text_val_df),
    desc="Text RAG Validation 평가"
):

    claim = row.text

    if claim not in val_text_cache:

        baseline = text_baseline_predict(claim)
        rag = text_rag_predict(claim, k=5)

        val_text_cache[claim] = {
            "baseline": baseline["prediction"],
            "rag": rag["prediction"],
            "top1_similarity": rag["neighbors"][0]["similarity"]
        }

    result = val_text_cache[claim]

    text_validation_results.append({
        "text": claim,
        "ground_truth": row.truth,
        "baseline_prediction": result["baseline"],
        "rag_prediction": result["rag"],
        "top1_similarity": result["top1_similarity"]
    })

    if len(text_validation_results) % 50 == 0:
        pd.DataFrame(text_validation_results).to_csv(
            "/content/text_validation_results.csv",
            index=False
        )

text_val_results = pd.DataFrame(text_validation_results)

text_val_results.to_csv(
    "/content/text_validation_results.csv",
    index=False
)

print("평가 완료:", len(text_val_results))
print("고유 문장 수:", len(val_text_cache))


# 성능 비교
summary = []

for name, col in [
    ("일반 Qwen", "baseline_prediction"),
    ("Text RAG + Qwen", "rag_prediction")
]:

    summary.append({
        "Model": name,
        "Accuracy": (
            text_val_results["ground_truth"]
            == text_val_results[col]
        ).mean(),
        "F1-score": f1_score(
            text_val_results["ground_truth"],
            text_val_results[col],
            labels=["FALSE"],
            average="macro",
            zero_division=0
        ),
        "Unknown Rate": (
            text_val_results[col] == "UNKNOWN"
        ).mean()
    })

display(pd.DataFrame(summary))

In [ ]:
train_text_set = set(
    text_train_df["text"].apply(
        lambda x: " ".join(x.casefold().split())
    )
)

text_val_results["duplicate"] = (
    text_val_results["text"]
    .apply(lambda x: " ".join(x.casefold().split()))
    .isin(train_text_set)
)

text_val_results["baseline_correct"] = (
    text_val_results["ground_truth"]
    == text_val_results["baseline_prediction"]
)

text_val_results["rag_correct"] = (
    text_val_results["ground_truth"]
    == text_val_results["rag_prediction"]
)

print("동일 문장 수:",
      text_val_results["duplicate"].sum())

print("유사도 0.95 이상:",
      (text_val_results["top1_similarity"] >= 0.95).sum())

print("\n중복 여부에 따른 정확도")

display(
    text_val_results.groupby("duplicate")[
        ["baseline_correct", "rag_correct"]
    ].mean()
)

In [ ]:
import numpy as np
import pandas as pd

y_true = text_val_results["ground_truth"].to_numpy()
baseline_preds = text_val_results["baseline_prediction"].to_numpy()
rag_preds = text_val_results["rag_prediction"].to_numpy()
similarities = text_val_results["top1_similarity"].to_numpy()

search_results = []

for threshold in np.r_[0.0, np.arange(0.50, 1.01, 0.01), 1.01]:

    use_rag = similarities >= threshold

    final_preds = np.where(
        use_rag,
        rag_preds,
        baseline_preds
    )

    search_results.append({
        "threshold": threshold,
        "accuracy": np.mean(final_preds == y_true),
        "rag_ratio": use_rag.mean(),
        "rag_count": use_rag.sum()
    })

text_grid_results = pd.DataFrame(search_results)

text_grid_results = text_grid_results.sort_values(
    ["accuracy", "rag_ratio"],
    ascending=[False, True]
).reset_index(drop=True)

best_text = text_grid_results.iloc[0]

TEXT_RAG_THRESHOLD = float(best_text["threshold"])

print("===== 최적 임계값 =====")
print(f"유사도 임계값: {TEXT_RAG_THRESHOLD:.2f}")

print("\n===== Validation 성능 =====")
print("일반 Qwen:", np.mean(baseline_preds == y_true))
print("일반 Text RAG:", np.mean(rag_preds == y_true))
print("선택적 Text RAG:", best_text["accuracy"])

print(f"RAG 사용 비율: {best_text['rag_ratio']:.2%}")
print("RAG 사용 문장 수:", int(best_text["rag_count"]))

display(text_grid_results.head(10))

In [ ]:
from tqdm.auto import tqdm

TEXT_RAG_THRESHOLD = 0.91

text_rag_test_cache = {}

unique_claims = cross_df["text"].unique()

for claim in tqdm(
    unique_claims,
    desc="Text RAG 최종 평가"
):

    result = text_rag_predict(claim, k=5)

    text_rag_test_cache[claim] = {
        "prediction": result["prediction"],
        "similarity": result["neighbors"][0]["similarity"]
    }

# 기존 실험 결과에 텍스트 연결
results_df = results_df.drop(
    columns=["text"], errors="ignore"
).merge(
    cross_df[["item_id", "text"]],
    on="item_id",
    how="left",
    validate="one_to_one"
)

assert len(results_df) == 5000
assert results_df["text"].notna().all()

# Text RAG 결과 추가
results_df["text_rag_prediction"] = results_df["text"].map(
    lambda x: text_rag_test_cache[x]["prediction"]
)

results_df["text_similarity"] = results_df["text"].map(
    lambda x: text_rag_test_cache[x]["similarity"]
)

# 선택적 Text RAG
results_df["text_selective_prediction"] = np.where(
    results_df["text_similarity"] >= TEXT_RAG_THRESHOLD,
    results_df["text_rag_prediction"],
    results_df["text_prediction"]
)

results_df.to_csv(
    "/content/full_test_results.csv",
    index=False
)

print("전체 문항:", len(results_df))
print("텍스트 추론 횟수:", len(text_rag_test_cache))

print("\n===== Text Accuracy =====")

for col in [
    "text_prediction",
    "text_rag_prediction",
    "text_selective_prediction"
]:
    acc = (results_df[col] == results_df["text_truth"]).mean()
    print(f"{col}: {acc:.2%}")

print("\n선택적 RAG 사용 비율:",
      (results_df["text_similarity"] >= TEXT_RAG_THRESHOLD).mean())

In [ ]:
import pandas as pd
import numpy as np
from sklearn.metrics import f1_score

experiments = {
    "A_Baseline": (
        "text_prediction", "baseline_image"
    ),
    "B_Image_RAG_GenD": (
        "text_prediction", "hybrid_image"
    ),
    "C_Text_RAG": (
        "text_rag_prediction", "baseline_image"
    ),
    "D_Full_Hybrid": (
        "text_rag_prediction", "hybrid_image"
    ),
    "E_Selective_Hybrid": (
        "text_selective_prediction", "hybrid_image"
    )
}

summary_rows = []

for name, (text_col, image_col) in experiments.items():

    # Text AND Image
    results_df[name] = [
        final_decision(t, i)
        for t, i in zip(
            results_df[text_col],
            results_df[image_col]
        )
    ]

    truth = results_df["final_truth"]
    pred = results_df[name]

    true_acc = (
        pred[truth == "TRUE"] == "TRUE"
    ).mean()

    false_acc = (
        pred[truth == "FALSE"] == "FALSE"
    ).mean()

    summary_rows.append({
        "Model": name,
        "Text Accuracy": (
            results_df[text_col] == results_df["text_truth"]
        ).mean(),
        "Image Accuracy": (
            results_df[image_col] == results_df["image_truth"]
        ).mean(),
        "Final Accuracy": (pred == truth).mean(),
        "F1-score": f1_score(
            truth, pred,
            labels=["FALSE"],
            average="macro",
            zero_division=0
        ),
        "Balanced Accuracy": (true_acc + false_acc) / 2,
        "FPR": (
            pred[truth == "TRUE"] == "FALSE"
        ).mean()
    })

final_summary = pd.DataFrame(summary_rows).set_index("Model")

print("===== 최종 모델 비교 (%) =====")
display((final_summary * 100).round(2))


# 네 가지 경우별 정확도
case_summary = pd.DataFrame()

for name in experiments:
    correct = results_df[name] == results_df["final_truth"]

    case_summary[name] = correct.groupby([
        results_df["text_truth"],
        results_df["image_truth"]
    ]).mean()

print("===== 경우별 정확도 (%) =====")
display((case_summary * 100).round(2))


# 데이터셋별 정확도
split_summary = pd.DataFrame()

for name in experiments:
    correct = results_df[name] == results_df["final_truth"]

    split_summary[name] = correct.groupby(
        results_df["split"]
    ).mean()

print("===== 데이터셋별 정확도 (%) =====")
display((split_summary * 100).round(2))


# 결과 저장
results_df.to_csv("/content/final_predictions.csv", index=False)
final_summary.to_csv("/content/final_summary.csv")
case_summary.to_csv("/content/case_summary.csv")
split_summary.to_csv("/content/split_summary.csv")

print("최종 실험 결과 저장 완료")

In [ ]:
import matplotlib.pyplot as plt

(final_summary[[
    "Text Accuracy",
    "Image Accuracy",
    "Final Accuracy"
]] * 100).plot(
    kind="bar",
    figsize=(13, 6),
    rot=15
)

plt.ylabel("Accuracy (%)")
plt.title("Deepfake Detection Model Comparison")
plt.ylim(0, 100)
plt.legend(loc="upper left")
plt.grid(axis="y", alpha=0.2)
plt.tight_layout()
plt.savefig("/content/final_comparison.png", dpi=300)
plt.show()

In [ ]:
import subprocess
import shutil
from pathlib import Path

REPO = Path("/content/DeepLearning_6team")
BRANCH = "qwen-mirage-experiment"

def git(*args):
    subprocess.run(
        ["git", *args],
        cwd=REPO,
        check=True
    )

# 새 브랜치 생성
git("switch", "-c", BRANCH)

# 실험 결과 저장 폴더
OUTPUT = REPO / "results" / "qwen_experiment"
OUTPUT.mkdir(parents=True, exist_ok=True)

# 실험 결과 복사
for filename in [
    "final_summary.csv",
    "case_summary.csv",
    "split_summary.csv",
    "final_comparison.png"
]:
    shutil.copy2(
        Path("/content") / filename,
        OUTPUT / filename
    )

# 원본 뉴스 문장을 제외한 판정 결과 저장
results_df.drop(
    columns=["text"],
    errors="ignore"
).to_csv(
    OUTPUT / "final_predictions.csv",
    index=False
)

git("branch", "--show-current")
git("status", "--short")

print("\n===== 저장된 파일 =====")

for path in OUTPUT.iterdir():
    print(path.name)

In [ ]:
from google.colab import files
import json

uploaded = files.upload()

for filename, content in uploaded.items():
    if filename.endswith(".ipynb"):

        notebook = json.loads(content.decode("utf-8"))

        # 실행 결과 및 출력 이미지 제거
        for cell in notebook["cells"]:
            if cell["cell_type"] == "code":
                cell["outputs"] = []
                cell["execution_count"] = None

        save_path = OUTPUT / "experiment.ipynb"

        save_path.write_text(
            json.dumps(notebook, ensure_ascii=False, indent=2),
            encoding="utf-8"
        )

        print("노트북 저장 완료:", save_path)

In [ ]:
readme = """# Qwen 기반 뉴스 진위 판별 실험

## 1. 실험 목적

일반 Qwen 모델과 RAG 및 GenD를 적용한 모델의
텍스트·이미지 진위 판별 성능을 비교한다.

텍스트와 이미지가 모두 TRUE일 때만 최종 TRUE로 판정한다.

## 2. 실험 환경

- Google Colab / Tesla T4
- PyTorch 2.14.1
- Transformers 5.24.1
- Qwen3-VL-2B-Instruct (4-bit NF4)
- CLIP ViT-B/32
- GenD CLIP-L/14
- all-MiniLM-L6-v2
- FAISS

## 3. 데이터셋

MiRAGeNews 데이터셋을 사용한다.

팀 저장소의 data_preparation/prepare_all.py로 다운로드한다.

- Image Train: 8,958장
- Image Validation: 2,344장
- Image Test: 2,500장
- Text Train: 10,000개
- Text Validation: 2,500개
- 통합 Test: 5,000문항

이미지는 256x256 정규화 버전을 사용했다.

## 4. 실험 방법

| Model | Text | Image |
|---|---|---|
| A | Qwen | Qwen |
| B | Qwen | Image RAG + GenD |
| C | Text RAG + Qwen | Qwen |
| D | Text RAG + Qwen | Image RAG + GenD |
| E | Selective Text RAG + Qwen | Image RAG + GenD |

### Image RAG

- CLIP 기반 512차원 이미지 임베딩
- FAISS 코사인 유사도 검색
- Top-5 라벨 다수결
- 유사도 임계값: 0.64
- 라벨 일치율: 0.60
- 조건 미충족 시 GenD 사용
- GenD 임계값: 0.28

### Text RAG

- MiniLM 기반 384차원 텍스트 임베딩
- FAISS Top-5 검색
- 검색된 문장, 라벨, 유사도를 Qwen에 전달
- 선택적 RAG 유사도 임계값: 0.91

모든 임계값은 Validation 데이터에서 결정했다.

## 5. 실험 결과

| Model | Text Acc | Image Acc | Final Acc | F1 | Balanced Acc | FPR |
|---|---:|---:|---:|---:|---:|---:|
| A | 64.56% | 53.64% | 68.58% | 78.75% | 59.53% | 58.56% |
| B | 64.56% | 63.76% | 74.10% | 83.95% | 57.85% | 74.64% |
| C | 56.52% | 53.64% | 72.08% | 82.90% | 53.95% | 82.32% |
| D | 56.52% | 63.76% | 74.68% | 85.03% | 53.49% | 88.88% |
| E | 64.64% | 63.76% | 74.14% | 83.98% | 57.93% | 74.48% |

![Experiment Results](final_comparison.png)

## 6. 결과 분석

- Image RAG + GenD 적용 시 이미지 정확도가
  53.64%에서 63.76%로 증가했다.
- Text RAG 적용 시 텍스트 정확도가
  64.56%에서 56.52%로 감소했다.
- 전체 적용 모델의 최종 정확도는
  68.58%에서 74.68%로 증가했다.
- 그러나 정상 콘텐츠에 대한 오탐률도 증가했다.
- 최종 FALSE 라벨의 비율이 75%이므로
  Accuracy만으로 성능 개선을 판단하기 어렵다.
- GenD 단독의 Validation AUC는 0.5075로 나타났다.

## 7. 한계

- Text RAG는 독립적인 사실 검증 근거가 아닌
  유사한 학습 문장과 라벨을 검색한다.
- GenD는 얼굴 딥페이크 탐지에 특화되어 있다.
- GenD의 개별 Test 성능은 측정하지 않았다.
- 데이터 분할 간 동일 사건 중복 여부는 추가 검증이 필요하다.
- 정규화 이미지에 대해서만 실험했다.
- 외부 웹 검색은 사용하지 않았다.

## 8. 파일 구성

- experiment.ipynb: Colab 실험 코드
- final_predictions.csv: 문항별 예측 결과
- final_summary.csv: 전체 성능
- case_summary.csv: 네 가지 진위 조합별 성능
- split_summary.csv: 데이터셋별 성능
- final_comparison.png: 비교 그래프

원본 데이터셋 및 모델 가중치는 저장소에 포함하지 않는다.
"""

(OUTPUT / "README.md").write_text(
    readme,
    encoding="utf-8"
)

print("README.md 생성 완료")

In [ ]:
import shutil
import subprocess
import tempfile
from pathlib import Path

BRANCH = "qwen-mirage-experiment"

# 완전히 새로운 Git 저장소 생성
NEW_REPO = Path(tempfile.mkdtemp(
    prefix="qwen_experiment_",
    dir="/content"
))

def new_git(*args):
    subprocess.run(
        ["git", *args],
        cwd=NEW_REPO,
        check=True
    )

new_git("init", "-b", BRANCH)

# 오늘 실험한 파일만 복사
filenames = [
    "README.md",
    "experiment.ipynb",
    "final_predictions.csv",
    "final_summary.csv",
    "case_summary.csv",
    "split_summary.csv",
    "final_comparison.png"
]

for filename in filenames:
    shutil.copy2(OUTPUT / filename, NEW_REPO / filename)

# 팀 GitHub 저장소 연결
new_git(
    "remote", "add", "origin",
    "https://github.com/Grow22/DeepLearning_6team.git"
)

new_git("add", ".")
new_git("status", "--short")

print("\n독립 저장소:", NEW_REPO)
print("브랜치:", BRANCH)

print("\n===== 포함된 파일 =====")
for filename in filenames:
    print(filename)